Bangla Agriculture Chatbot: Llama 3.2 1B Instruct Fine-Tuning

Install libraries

In [ ]:
!pip install -q -U \
    "transformers>=4.48,<5" \
    "datasets>=3.0" \
    "accelerate>=1.0" \
    "peft>=0.14" \
    "trl>=0.24,<1" \
    "bitsandbytes>=0.45" \
    "huggingface_hub>=0.27" \
    "bert-score>=0.3.13" \
    "rouge-score>=0.1.2" \
    "nltk>=3.9" \
    "pandas>=2.0" \
    "tqdm>=4.66"


Imports, reproducibility, and GPU check

In [ ]:
import os
import gc
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "A CUDA GPU is required for this QLoRA notebook. "
        "In Colab choose Runtime > Change runtime type > GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())


Hugging Face login

In [ ]:
from huggingface_hub import notebook_login

notebook_login("huggingface_token.txt")


Configuration

In [ ]:

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"


TRAIN_URL = (
    "https://raw.githubusercontent.com/"
    "RamijWasithRahat/Bangla-Agriculture-Chatbot/"
    "refs/heads/main/Data/Bangla_Agriculture_QA_Train_800.json"
)

TEST_URL = (
    "https://raw.githubusercontent.com/"
    "RamijWasithRahat/Bangla-Agriculture-Chatbot/"
    "refs/heads/main/Data/Bangla_Agriculture_QA_Test_200.json"
)


OUTPUT_DIR = "./llama-3.2-1b-bangla-agriculture-qlora"
ADAPTER_DIR = "./llama-3.2-1b-bangla-agriculture-adapter"

RESULT_JSON_PATH = "./llama_3_2_1b_finetuned_test_results.json"
SUMMARY_JSON_PATH = "./llama_3_2_1b_finetuned_summary.json"
RESULT_CSV_PATH = "./llama_3_2_1b_finetuned_test_results.csv"
COMPARISON_CSV_PATH = "./llama_3_2_1b_baseline_vs_finetuned.csv"


MAX_LENGTH = 512

NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 2e-4


GENERATION_BATCH_SIZE = 8
MAX_NEW_TOKENS = 160

SYSTEM_PROMPT = (
    "তুমি বাংলাদেশের কৃষি বিষয়ক প্রশ্নের উত্তর দেওয়ার জন্য একটি সহায়ক সহকারী। "
    "শুধু প্রশ্নের প্রাসঙ্গিক উত্তর বাংলায় দাও। "
    "যেখানে সংখ্যা, সময়, দূরত্ব, পরিমাণ বা নির্দিষ্ট তথ্য আছে সেখানে তা সঠিকভাবে উল্লেখ করো। "
    "অপ্রয়োজনীয় তথ্য তৈরি করো না।"
)

print("MODEL_ID =", MODEL_ID)


Load 800 training and 200 test examples

In [ ]:
train_full_raw = load_dataset(
    "json",
    data_files=TRAIN_URL,
    field="qa_pairs",
    split="train",
)

test_raw = load_dataset(
    "json",
    data_files=TEST_URL,
    field="qa_pairs",
    split="train",
)

print(train_full_raw)
print(test_raw)

assert len(train_full_raw) == 800, (
    f"Expected 800 training examples, got {len(train_full_raw)}"
)
assert len(test_raw) == 200, (
    f"Expected 200 test examples, got {len(test_raw)}"
)
assert set(["id", "question", "reference_answer"]).issubset(train_full_raw.column_names)
assert set(["id", "question", "reference_answer"]).issubset(test_raw.column_names)

print("\nFirst training record:")
print(train_full_raw[0])

print("\nFirst test record:")
print(test_raw[0])


Validate the data: check non-empty strings

In [ ]:
def validate_dataset(ds, name):
    problems = []

    for i, row in enumerate(ds):
        q = row.get("question")
        a = row.get("reference_answer")

        if not isinstance(q, str) or not q.strip():
            problems.append((i, "question"))

        if not isinstance(a, str) or not a.strip():
            problems.append((i, "reference_answer"))

    print(f"{name}: {len(ds)} rows")
    print(f"{name}: invalid fields = {len(problems)}")

    if problems:
        raise ValueError(f"{name} contains invalid rows: {problems[:10]}")


validate_dataset(train_full_raw, "TRAIN")
validate_dataset(test_raw, "TEST")


Use all 800 training examples

In [ ]:
train_raw = train_full_raw

print("Fine-tuning examples:", len(train_raw))
print("Untouched final test examples:", len(test_raw))

assert len(train_raw) == 800
assert len(test_raw) == 200


Convert QA pairs to Hugging Face prompt-completion format

In [ ]:
def to_sft_format(example):
    return {
        "prompt": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": example["question"].strip(),
            },
        ],
        "completion": [
            {
                "role": "assistant",
                "content": example["reference_answer"].strip(),
            }
        ],
    }


train_sft = train_raw.map(
    to_sft_format,
    remove_columns=train_raw.column_names,
)

print(train_sft)
print("\nExample:")
print(train_sft[0])


Load the Llama 3.2 tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Right padding is convenient for training.
tokenizer.padding_side = "right"

print("EOS token:", tokenizer.eos_token)
print("EOS token id:", tokenizer.eos_token_id)
print("PAD token:", tokenizer.pad_token)
print("Chat template available:", tokenizer.chat_template is not None)


Check training sequence lengths

In [ ]:
def formatted_length(example):
    messages = example["prompt"] + example["completion"]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return len(
        tokenizer(
            text,
            add_special_tokens=False,
        )["input_ids"]
    )


lengths = [formatted_length(x) for x in train_sft]

print("Minimum length:", int(np.min(lengths)))
print("Median length:", int(np.median(lengths)))
print("95th percentile:", int(np.percentile(lengths, 95)))
print("Maximum length:", int(np.max(lengths)))
print(
    f"Examples longer than MAX_LENGTH={MAX_LENGTH}:",
    sum(x > MAX_LENGTH for x in lengths),
)


Configure 4-bit QLoRA quantization

In [ ]:
use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print("Compute dtype:", compute_dtype)


Load `meta-llama/Llama-3.2-1B-Instruct`

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=compute_dtype,
)

model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

print("Loaded:", MODEL_ID)
print("4-bit:", getattr(model, "is_loaded_in_4bit", False))


Configure LoRA adapter

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

print(peft_config)


Configure Hugging Face TRL `SFTTrainer`

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,


    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,

    max_length=MAX_LENGTH,
    completion_only_loss=True,
    packing=False,


    gradient_checkpointing=True,
    bf16=use_bf16,
    fp16=not use_bf16,
    optim="paged_adamw_8bit",


    save_strategy="epoch",
    save_total_limit=2,


    logging_steps=10,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
)

print(training_args)


Create the trainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_sft,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print("Trainer created.")
trainer.model.print_trainable_parameters()


Fine-tune Llama 3.2 1B

In [ ]:
train_result = trainer.train()
train_result


Training summary

In [ ]:
print("Training metrics:")
for key, value in train_result.metrics.items():
    print(f"{key}: {value}")


Save the fine-tuned LoRA adapter

In [ ]:
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Adapter saved to:", ADAPTER_DIR)


Deterministic generation function

In [ ]:

trainer.model.config.use_cache = True
trainer.model.eval()

terminators = [tokenizer.eos_token_id]

eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
if isinstance(eot_id, int) and eot_id >= 0 and eot_id not in terminators:
    terminators.append(eot_id)

print("Generation terminators:", terminators)


def make_generation_prompt(question):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": question.strip(),
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


Quick sanity check on one held-out test question

In [ ]:
def generate_one(question, max_new_tokens=MAX_NEW_TOKENS):
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    prompt = make_generation_prompt(question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")

    with torch.inference_mode():
        output = trainer.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=terminators,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = output[0, inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()

    tokenizer.padding_side = original_padding_side
    return answer


sample = test_raw[0]

print("Question:")
print(sample["question"])

print("\nReference:")
print(sample["reference_answer"])

print("\nFine-tuned prediction:")
print(generate_one(sample["question"]))


Generate predictions for all 200 test examples

In [ ]:
from tqdm.auto import tqdm


def generate_batch(questions, max_new_tokens=MAX_NEW_TOKENS):
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    prompts = [
        make_generation_prompt(question)
        for question in questions
    ]

    batch = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        add_special_tokens=False,
    ).to("cuda")

    input_width = batch["input_ids"].shape[1]

    with torch.inference_mode():
        outputs = trainer.model.generate(
            **batch,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=terminators,
            pad_token_id=tokenizer.pad_token_id,
        )

    predictions = []

    for output in outputs:
        generated = output[input_width:]

        text = tokenizer.decode(
            generated,
            skip_special_tokens=True,
        ).strip()

        predictions.append(text)

    tokenizer.padding_side = original_padding_side
    return predictions


all_predictions = []

for start in tqdm(
    range(0, len(test_raw), GENERATION_BATCH_SIZE),
    desc="Generating test predictions",
):
    end = min(
        start + GENERATION_BATCH_SIZE,
        len(test_raw),
    )

    questions = test_raw[start:end]["question"]

    batch_predictions = generate_batch(questions)
    all_predictions.extend(batch_predictions)


assert len(all_predictions) == len(test_raw) == 200

print("Generated predictions:", len(all_predictions))
print("\nFirst prediction:")
print(all_predictions[0])


Evaluation metrics calculation

Calculate BLEU, ROUGE-L, Exact Match, and BERTScore

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu
from rouge_score import rouge_scorer
from bert_score import score as bert_score

warnings.filterwarnings(
    "ignore",
    message="The hypothesis contains 0 counts of .*",
)

references = [
    str(x).strip()
    for x in test_raw["reference_answer"]
]

predictions = [
    str(x).strip()
    for x in all_predictions
]


sentence_bleu_scores = []

for reference, prediction in zip(references, predictions):
    score_value = sentence_bleu(
        [reference.split()],
        prediction.split(),
    )
    sentence_bleu_scores.append(float(score_value))




rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True,
)

rouge_l_scores = []

for reference, prediction in zip(references, predictions):
    rouge_value = rouge.score(
        reference,
        prediction,
    )["rougeL"].fmeasure

    rouge_l_scores.append(float(rouge_value))




exact_match_scores = [
    int(prediction == reference)
    for prediction, reference in zip(predictions, references)
]




P, R, F1 = bert_score(
    predictions,
    references,
    lang="bn",
    batch_size=16,
    verbose=True,
)

bert_precision_scores = [float(x) for x in P.cpu().numpy()]
bert_recall_scores = [float(x) for x in R.cpu().numpy()]
bert_f1_scores = [float(x) for x in F1.cpu().numpy()]




corpus_bleu_value = float(
    corpus_bleu(
        [[reference.split()] for reference in references],
        [prediction.split() for prediction in predictions],
    )
)

print("Metric calculation complete.")


Build the requested per-question JSON result

In [ ]:
results = []

for i in range(len(test_raw)):
    row = {
        "id": int(test_raw[i]["id"]),
        "question": test_raw[i]["question"],
        "reference_answer": test_raw[i]["reference_answer"],
        "predicted_answer": predictions[i],
        "sentence_bleu": sentence_bleu_scores[i],
        "rouge_score": rouge_l_scores[i],
        "exact_match": exact_match_scores[i],
        "bert_score_precision": bert_precision_scores[i],
        "bert_score_recall": bert_recall_scores[i],
        "bert_score_f1": bert_f1_scores[i],
    }

    results.append(row)


print(json.dumps(
    results[:3],
    ensure_ascii=False,
    indent=2,
))


Save the full 200-question JSON and CSV

In [ ]:
with open(
    RESULT_JSON_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2,
    )


results_df = pd.DataFrame(results)



print("Saved JSON:", RESULT_JSON_PATH)
print("Rows:", len(results_df))


Print summary in baseline format

In [ ]:
average_bleu = float(np.mean(sentence_bleu_scores))
average_rouge_l = float(np.mean(rouge_l_scores))
average_bert_f1 = float(np.mean(bert_f1_scores))
exact_match_accuracy = float(np.mean(exact_match_scores))

summary = {
    "model": MODEL_ID,
    "evaluation": "Fine-Tuned",
    "num_test_examples": len(test_raw),
    "average_bleu_score": average_bleu,
    "average_rouge_l_score": average_rouge_l,
    "average_bertscore_f1": average_bert_f1,
    "exact_match_accuracy": exact_match_accuracy,
    "corpus_bleu_score": corpus_bleu_value,
}

print("=" * 70)
print("Llama 3.2 1B Instruct Fine-Tuned")
print("=" * 70)
print("Average BLEU Score:", average_bleu)
print("Average ROUGE-L Score:", average_rouge_l)
print("Average BERTScore F1:", average_bert_f1)
print("Exact Match Accuracy:", exact_match_accuracy)
print("Corpus BLEU Score:", corpus_bleu_value)


Save the fine-tuned summary JSON

In [ ]:
with open(
    SUMMARY_JSON_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(json.dumps(
    summary,
    ensure_ascii=False,
    indent=2,
))

print("\nSaved:", SUMMARY_JSON_PATH)


Baseline vs Fine-Tuned Comparison

Compare the fine-tuned model against your baseline

In [ ]:
baseline_summary = {
    "Average BLEU Score": 0.21769990715331805,
    "Average ROUGE-L Score": 0.0,
    "Average BERTScore F1": 0.6948550343513489,
    "Exact Match Accuracy": 0.0,
    "Corpus BLEU Score": 0.27344330427678804,
}

finetuned_summary = {
    "Average BLEU Score": average_bleu,
    "Average ROUGE-L Score": average_rouge_l,
    "Average BERTScore F1": average_bert_f1,
    "Exact Match Accuracy": exact_match_accuracy,
    "Corpus BLEU Score": corpus_bleu_value,
}

comparison_rows = []

for metric in baseline_summary:
    baseline_value = baseline_summary[metric]
    finetuned_value = finetuned_summary[metric]

    comparison_rows.append({
        "metric": metric,
        "baseline": baseline_value,
        "fine_tuned": finetuned_value,
        "absolute_change": finetuned_value - baseline_value,
    })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df


In [ ]:
comparison_df.to_csv(
    COMPARISON_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", COMPARISON_CSV_PATH)


Bengali-aware ROUGE-L diagnostic

In [ ]:
def lcs_length(a, b):

    previous = [0] * (len(b) + 1)

    for token_a in a:
        current = [0]

        for j, token_b in enumerate(b, start=1):
            if token_a == token_b:
                current.append(previous[j - 1] + 1)
            else:
                current.append(
                    max(previous[j], current[-1])
                )

        previous = current

    return previous[-1]


def bangla_rouge_l_f1(reference, prediction):
    ref_tokens = reference.split()
    pred_tokens = prediction.split()

    if not ref_tokens or not pred_tokens:
        return 0.0

    lcs = lcs_length(ref_tokens, pred_tokens)

    if lcs == 0:
        return 0.0

    precision = lcs / len(pred_tokens)
    recall = lcs / len(ref_tokens)

    return (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )


bangla_rouge_scores = [
    bangla_rouge_l_f1(reference, prediction)
    for reference, prediction in zip(references, predictions)
]

print(
    "Bengali-aware Average ROUGE-L F1:",
    float(np.mean(bangla_rouge_scores)),
)


Inspect the best and worst predictions

In [ ]:
inspection_df = results_df.copy()

inspection_df = inspection_df.sort_values(
    "bert_score_f1",
    ascending=False,
)

print("Top 5 by BERTScore F1:")
display(
    inspection_df[
        [
            "id",
            "question",
            "reference_answer",
            "predicted_answer",
            "bert_score_f1",
        ]
    ].head(5)
)

print("\nBottom 5 by BERTScore F1:")
display(
    inspection_df[
        [
            "id",
            "question",
            "reference_answer",
            "predicted_answer",
            "bert_score_f1",
        ]
    ].tail(5)
)
